In [1]:
!pip install HEBO -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.9/795.9 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.7/277.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.7 MB/s eta 0:00:00
  

In [2]:
!pip install botorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.9/779.9 kB 516.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.5 MB/s eta 0:00:00


In [3]:
# how to optimize with hebo: https://github.com/huawei-noah/HEBO/blob/master/HEBO/doc/source/optimisation.ipynb
from hebo.optimizers.hebo import HEBO
from hebo.design_space.design_space import DesignSpace


In [4]:
from botorch.test_functions import Ackley


In [5]:
import torch

dtype = torch.double

In [6]:
from tqdm import tqdm


def test_hebo(budget, init_n=4, q=1):
    # setup problem (fast)
    a6_problem = Ackley(dim=6, noise_std=None, negate=True)  # assume maximization problem
    lbs = a6_problem.bounds[0, :]
    ubs = a6_problem.bounds[1, :]
    params = [
        {'name': f'x{i}', 'type': 'num', 'lb': lbi, 'ub': ubi} 
        for i, lbi, ubi in zip(range(6), lbs.detach().numpy(), ubs.detach().numpy())]
    hebo_design_space = DesignSpace().parse(params)
    # init optimizer
    optimizer = HEBO(hebo_design_space, rand_sample=init_n)
    # optimize
    best_results = []
    trials_spend = 0
    with tqdm(total=budget, leave=False)as pbar:
        while trials_spend < budget:
            cand = optimizer.suggest(n_suggestions=1)
            x = torch.from_numpy(cand.iloc[-1].to_numpy())
            y = a6_problem(x)
            optimizer.observe(cand, -1 * y.detach().numpy().reshape(-1))  # negate for optimizing for minimization problem

            # current_best_max = -optimizer.y.min()
            if best_results:
                best_results.append(max(best_results[-1], y.item())) 
            else:
                best_results.append(y.item())
            pbar.update(1)
            trials_spend += 1

    return best_results


def conduct_experiment(n_repeats, budget=100, init_n=1, q=1):
    all_res = []
    for i in tqdm(range(n_repeats), total=n_repeats):
        res = test_hebo(budget=budget, init_n=init_n, q=q)
        all_res.append(res)
    return all_res



In [7]:
vanilla_hebo_res = conduct_experiment(n_repeats=30, budget=100, init_n=4, q=1)  # 1 repeat - 9 min,

  5%|▌         | 5/100 [00:03<01:01,  1.55it/s]

jitter = 1e-07



  0%|          | 0/100 [00:00<?, ?it/s]

jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fittin


  5%|▌         | 5/100 [00:06<02:01,  1.28s/it]

jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions



  5%|▌         | 5/100 [00:03<01:01,  1.53it/s]

jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07



  5%|▌         | 5/100 [00:03<01:07,  1.41it/s]

jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07



100%|██████████| 30/30 [2:42:25<00:00, 324.84s/it]


In [8]:
# vanilla_hebo_res

In [9]:
import numpy as np


np.savetxt("/kaggle/working/vanilla_hebo_res_A6.txt", np.array(vanilla_hebo_res))